In [ ]:
# ============================================================
# STEP 4 - CLUSTERING FROM DGI NODE EMBEDDINGS
# Original vs synthetic DGI embeddings
# + post-training vs training-time comparison
# KMeans / GMM / Spectral + interpretation
# ============================================================

# ============================================================
# CELL 1 - Imports
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns

from sklearn.cluster import KMeans, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)
from sklearn.manifold import TSNE
from sklearn.feature_selection import mutual_info_classif
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

# ============================================================
# ADDITIONAL EXPERIMENT
# ============================================================
# Original Graph, One Encoder, and Batch are evaluated with:
#   (a) current post-training/re-encoded embeddings
#   (b) embeddings produced directly during training
#
# Multiple Encoders remains unchanged as the reference approach.
# ============================================================


In [ ]:
# ============================================================
# CELL 2 - Load embeddings and matrices
# ============================================================

def load_embeddings(embeddings_path):
    return np.load(embeddings_path)


def load_matrix_for_synthetic_order(
    path,
    reference_node_ids=None,
    matrix_name="matrix"
):
    """
    Loads a node-level matrix and reindexes it to the positional node order
    1..n used by the synthetic graphs.

    If reference_node_ids is supplied, the matrix is explicitly aligned
    to that original-node order before converting to positions 1..n.
    """

    df_original = pd.read_csv(path, index_col=0)

    df_original.index = df_original.index.astype(int)
    df_original.index.name = "node_id"

    if reference_node_ids is not None:
        reference_node_ids = [int(node_id) for node_id in reference_node_ids]

        missing_nodes = set(reference_node_ids) - set(df_original.index)
        extra_nodes = set(df_original.index) - set(reference_node_ids)

        if missing_nodes:
            raise ValueError(
                f"{matrix_name}: missing nodes relative to ARD order: "
                f"{sorted(missing_nodes)[:10]}"
            )

        if extra_nodes:
            raise ValueError(
                f"{matrix_name}: extra nodes not present in ARD order: "
                f"{sorted(extra_nodes)[:10]}"
            )

        df_original = df_original.loc[reference_node_ids]

    original_node_ids = df_original.index.tolist()

    df = df_original.reset_index(drop=True)
    df.index = range(1, len(df) + 1)
    df.index.name = "node_position"

    return df, original_node_ids


def plot_feature_sparsity(features_df):
    X = features_df.to_numpy()
    feature_means = X.mean(axis=0)

    plt.figure(figsize=(10, 6))
    plt.hist(feature_means, bins=20, edgecolor="black", alpha=0.7)
    plt.title("Histogram of Mean Feature Values")
    plt.xlabel("Mean feature value")
    plt.ylabel("Number of features")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return feature_means

In [ ]:
# ============================================================
# CELL 3 - Clustering diagnostics from k=2 to k=8
# ============================================================

def compute_internal_clustering_metrics(embeddings, labels):
    """
    Computes internal clustering metrics safely.

    If the fitted method produces fewer than two effective clusters,
    internal clustering metrics are undefined and returned as NaN.
    """

    n_samples = len(labels)
    n_clusters_found = len(np.unique(labels))

    if n_clusters_found < 2 or n_clusters_found >= n_samples:
        return {
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
            "n_clusters_found": n_clusters_found,
            "valid_internal_metrics": False,
        }

    return {
        "silhouette": silhouette_score(embeddings, labels),
        "calinski_harabasz": calinski_harabasz_score(
            embeddings,
            labels
        ),
        "davies_bouldin": davies_bouldin_score(
            embeddings,
            labels
        ),
        "n_clusters_found": n_clusters_found,
        "valid_internal_metrics": True,
    }


def fit_clustering_method_safely(
    method_name,
    embeddings,
    k,
    random_state
):
    """
    Fits a clustering method safely.

    A collapsed result or fitting error is recorded rather than
    interrupting the diagnostics for the remaining datasets.
    """

    try:
        if method_name == "kmeans":
            model = KMeans(
                n_clusters=k,
                random_state=random_state,
                n_init=10
            )

        elif method_name == "gmm":
            model = GaussianMixture(
                n_components=k,
                covariance_type="diag",
                reg_covar=1e-4,
                n_init=5,
                random_state=random_state
            )

        elif method_name == "spectral":
            model = SpectralClustering(
                n_clusters=k,
                affinity="rbf",
                random_state=random_state
            )

        else:
            raise ValueError(f"Unknown method: {method_name}")

        labels = model.fit_predict(embeddings)

        metrics = compute_internal_clustering_metrics(
            embeddings,
            labels
        )

        metrics["method"] = method_name
        metrics["k"] = k
        metrics["fit_status"] = "ok"

        if not metrics["valid_internal_metrics"]:
            metrics["fit_status"] = (
                "collapsed_or_invalid_number_of_clusters"
            )

        return metrics

    except Exception as error:
        return {
            "method": method_name,
            "k": k,
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
            "n_clusters_found": np.nan,
            "valid_internal_metrics": False,
            "fit_status": f"error: {type(error).__name__}",
        }


def compute_clustering_diagnostics(
    embeddings,
    k_min=2,
    k_max=8,
    random_state=123
):
    results = []

    for k in range(k_min, k_max + 1):

        for method_name in ["kmeans", "gmm", "spectral"]:
            result = fit_clustering_method_safely(
                method_name=method_name,
                embeddings=embeddings,
                k=k,
                random_state=random_state
            )

            results.append(result)

    return pd.DataFrame(results)


def split_diagnostics_by_method(diagnostics_df, output_folder, dataset_name):
    diagnostics_by_method = {}

    for method in diagnostics_df["method"].unique():
        df_method = diagnostics_df[
            diagnostics_df["method"] == method
        ].copy()

        diagnostics_by_method[method] = df_method

        df_method.to_csv(
            os.path.join(
                output_folder,
                f"{dataset_name}_diagnostics_{method}_k2_k8.csv"
            ),
            index=False
        )

        print(f"\n=== {dataset_name} | {method.upper()} diagnostics ===")
        print(df_method)

    return diagnostics_by_method

In [ ]:
# ============================================================
# CELL 4 - Clustering models
# ============================================================

def evaluate_clustering(embeddings, labels):
    """
    Evaluates a fitted clustering safely.

    If the method identifies only one effective cluster,
    internal metrics are undefined and returned as NaN.
    """

    n_clusters_found = len(np.unique(labels))
    n_samples = len(labels)

    if n_clusters_found < 2 or n_clusters_found >= n_samples:
        return {
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
            "n_clusters": n_clusters_found,
            "valid_internal_metrics": False,
            "fit_status": "collapsed_or_invalid_number_of_clusters",
        }

    return {
        "silhouette": silhouette_score(embeddings, labels),
        "calinski_harabasz": calinski_harabasz_score(
            embeddings,
            labels
        ),
        "davies_bouldin": davies_bouldin_score(
            embeddings,
            labels
        ),
        "n_clusters": n_clusters_found,
        "valid_internal_metrics": True,
        "fit_status": "ok",
    }


def run_kmeans(embeddings, n_clusters, random_state=123):
    model = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=10
    )

    labels = model.fit_predict(embeddings)
    metrics = evaluate_clustering(embeddings, labels)
    centroides = model.cluster_centers_

    return labels, metrics, model, centroides


def run_gmm(embeddings, n_clusters, random_state=123):
    model = GaussianMixture(
        n_components=n_clusters,
        covariance_type="diag",
        reg_covar=1e-4,
        n_init=5,
        random_state=random_state
    )

    labels = model.fit_predict(embeddings)
    metrics = evaluate_clustering(embeddings, labels)

    return labels, metrics, model


def run_spectral(embeddings, n_clusters, random_state=123):
    model = SpectralClustering(
        n_clusters=n_clusters,
        affinity="rbf",
        random_state=random_state,
    )

    labels = model.fit_predict(embeddings)
    metrics = evaluate_clustering(embeddings, labels)

    return labels, metrics, model


def make_failed_clustering_result(error_message):
    return {
        "labels": None,
        "metrics": {
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
            "n_clusters": np.nan,
            "valid_internal_metrics": False,
            "fit_status": error_message,
        },
        "model": None,
    }


def compare_clustering_methods(embeddings, n_clusters, random_state=123):
    results = {}

    try:
        labels_kmeans, metrics_kmeans, model_kmeans, centroides = run_kmeans(
            embeddings,
            n_clusters,
            random_state
        )

        results["kmeans"] = {
            "labels": labels_kmeans,
            "metrics": metrics_kmeans,
            "model": model_kmeans,
            "centroides": centroides,
        }

    except Exception as error:
        results["kmeans"] = make_failed_clustering_result(
            f"error: {type(error).__name__}"
        )
        results["kmeans"]["centroides"] = None

    try:
        labels_gmm, metrics_gmm, model_gmm = run_gmm(
            embeddings,
            n_clusters,
            random_state
        )

        results["gmm"] = {
            "labels": labels_gmm,
            "metrics": metrics_gmm,
            "model": model_gmm,
        }

    except Exception as error:
        results["gmm"] = make_failed_clustering_result(
            f"error: {type(error).__name__}"
        )

    try:
        labels_spectral, metrics_spectral, model_spectral = run_spectral(
            embeddings,
            n_clusters,
            random_state
        )

        results["spectral"] = {
            "labels": labels_spectral,
            "metrics": metrics_spectral,
            "model": model_spectral,
        }

    except Exception as error:
        results["spectral"] = make_failed_clustering_result(
            f"error: {type(error).__name__}"
        )

    metrics_df = pd.DataFrame({
        method: result["metrics"]
        for method, result in results.items()
    }).T

    return results, metrics_df

In [ ]:
# ============================================================
# CELL 5 - Visualization
# ============================================================

def compute_tsne(embeddings, random_state=123):
    tsne = TSNE(
        n_components=2,
        random_state=random_state,
        init="pca",
        learning_rate="auto"
    )

    emb_2d = tsne.fit_transform(embeddings)

    return emb_2d


def plot_tsne_clusters(emb_2d, labels, title="DGI Embeddings - t-SNE"):
    plt.figure(figsize=(8, 6))
    plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=labels, cmap="tab10")
    plt.title(title)
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.tight_layout()
    plt.show()


def plot_cluster_sizes(labels):
    cluster_counts = pd.Series(labels).value_counts().sort_index()

    plt.figure(figsize=(8, 5))
    sns.barplot(x=cluster_counts.index, y=cluster_counts.values, palette="Paired")
    plt.xlabel("Cluster ID")
    plt.ylabel("Number of Nodes")
    plt.title("Cluster Sizes")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    return cluster_counts

In [ ]:
# ============================================================
# CELL 6 - Save cluster labels
# ============================================================

def save_cluster_labels(labels, output_path):
    df = pd.DataFrame({
        "node_position": np.arange(1, len(labels) + 1),
        "cluster": labels,
    })

    df.to_csv(output_path, index=False)

    return df

In [ ]:
# ============================================================
# CELL 7 - Feature interpretation by cluster
# ============================================================

def compute_cluster_feature_profiles(features_df, labels):
    df = features_df.copy()
    df["cluster"] = labels

    cluster_feature_proportions = df.groupby("cluster").mean()

    return cluster_feature_proportions


def plot_feature_profile_heatmap(cluster_feature_proportions):
    plt.figure(figsize=(12, 8))
    sns.heatmap(
        cluster_feature_proportions.T,
        cmap="Blues",
        annot=True,
        fmt=".2f"
    )

    plt.title("Mean Feature Value Per Cluster")
    plt.xlabel("Cluster")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()


def print_features_above_threshold(cluster_feature_proportions, threshold=0.3):
    mask = cluster_feature_proportions > threshold
    features_above_threshold = mask.any(axis=0)

    features_over_threshold = cluster_feature_proportions.loc[:, features_above_threshold]

    sorted_features = features_over_threshold.loc[
        :,
        features_over_threshold.max() > threshold
    ]

    print(f"Features with > {int(threshold * 100)}% prevalence/value in at least one cluster:")
    print(sorted_features)

    return sorted_features


def compute_mutual_information_features(features_df, labels):
    X = features_df.to_numpy()
    y = labels

    mi = mutual_info_classif(
        X,
        y,
        discrete_features=True,
        random_state=123
    )

    mi_scores = pd.Series(mi, index=features_df.columns).sort_values(ascending=False)

    return mi_scores


def plot_top_features_per_cluster(cluster_feature_proportions, threshold=0.4):
    num_clusters = cluster_feature_proportions.shape[0]

    fig, axes = plt.subplots(
        num_clusters,
        1,
        figsize=(12, 5 * num_clusters),
        constrained_layout=True
    )

    if num_clusters == 1:
        axes = [axes]

    for i, cluster_id in enumerate(cluster_feature_proportions.index):
        prevalence = cluster_feature_proportions.loc[cluster_id].sort_values(ascending=False)
        top_prevalence = prevalence[prevalence > threshold]

        axes[i].bar(
            top_prevalence.index,
            top_prevalence.values,
            color="steelblue"
        )

        axes[i].set_title(
            f"Cluster {cluster_id} - Top Features > {threshold}"
        )

        axes[i].set_ylabel("Mean value")
        axes[i].tick_params(axis="x", rotation=45)

    plt.suptitle("Feature Values per Cluster", fontsize=16)
    plt.show()

In [ ]:
# ============================================================
# CELL 8 - CLAMP-style cluster prototypes
# ============================================================

def compute_clamp_feature_prototypes(
    features_df,
    labels,
    lower_q=0.10,
    upper_q=0.90,
    top_n=15
):
    df = features_df.copy()
    df["cluster"] = labels

    global_mean = features_df.mean(axis=0)
    global_std = features_df.std(axis=0)

    prototypes = {}

    for cluster_id, group in df.groupby("cluster"):
        X_cluster = group.drop(columns="cluster")

        cluster_mean = X_cluster.mean(axis=0)
        q_low = X_cluster.quantile(lower_q, axis=0)
        q_high = X_cluster.quantile(upper_q, axis=0)

        proto = pd.DataFrame({
            "cluster": cluster_id,
            "feature": features_df.columns,
            "cluster_prevalence": cluster_mean.values,
            "global_prevalence": global_mean.values,
            "prevalence_diff": (cluster_mean - global_mean).values,
            "abs_prevalence_diff": (cluster_mean - global_mean).abs().values,
            "lift": (cluster_mean / global_mean.replace(0, np.nan)).values,
            f"q{int(lower_q * 100)}": q_low.values,
            f"q{int(upper_q * 100)}": q_high.values,
            "global_std": global_std.values,
        })

        proto = proto.sort_values(
            "abs_prevalence_diff",
            ascending=False
        ).head(top_n)

        prototypes[cluster_id] = proto

    prototypes_df = pd.concat(prototypes.values(), axis=0).reset_index(drop=True)

    return prototypes_df


def plot_clamp_feature_prototypes(
    prototypes_df,
    top_n=10,
    value_col="cluster_prevalence"
):
    cluster_ids = sorted(prototypes_df["cluster"].unique())

    for cluster_id in cluster_ids:
        cluster_proto = prototypes_df[
            prototypes_df["cluster"] == cluster_id
        ].head(top_n).copy()

        cluster_proto = cluster_proto.sort_values(value_col, ascending=True)

        plt.figure(figsize=(10, max(4, 0.35 * len(cluster_proto))))

        plt.barh(
            cluster_proto["feature"],
            cluster_proto[value_col],
            alpha=0.8,
            label="Cluster mean"
        )

        plt.scatter(
            cluster_proto["global_prevalence"],
            cluster_proto["feature"],
            marker="x",
            s=70,
            label="Global mean"
        )

        plt.xlabel("Feature mean value")
        plt.ylabel("Feature")
        plt.title(f"CLAMP-style prototype - Cluster {cluster_id}")
        plt.legend()
        plt.tight_layout()
        plt.show()


def plot_clamp_heatmap(prototypes_df, top_n_features=30):
    selected_features = (
        prototypes_df
        .groupby("feature")["abs_prevalence_diff"]
        .max()
        .sort_values(ascending=False)
        .head(top_n_features)
        .index
    )

    heatmap_df = prototypes_df[
        prototypes_df["feature"].isin(selected_features)
    ].pivot_table(
        index="feature",
        columns="cluster",
        values="prevalence_diff",
        aggfunc="mean"
    ).fillna(0)

    plt.figure(figsize=(12, max(6, 0.35 * len(heatmap_df))))
    sns.heatmap(
        heatmap_df,
        cmap="coolwarm",
        center=0,
        annot=True,
        fmt=".2f"
    )

    plt.title("CLAMP-style feature differences by cluster")
    plt.xlabel("Cluster")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

    return heatmap_df

In [ ]:
# ============================================================
# CELL 9 - Decision tree explanation of clusters
# ============================================================

def train_decision_tree_cluster_explainer(
    features_df,
    labels,
    max_depth=4,
    min_samples_leaf=10,
    random_state=123
):
    X = features_df.copy()
    y = np.asarray(labels)

    tree = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=random_state,
        class_weight="balanced"
    )

    tree.fit(X, y)

    y_pred = tree.predict(X)

    metrics = {
        "tree_fidelity_accuracy": accuracy_score(y, y_pred),
        "tree_fidelity_balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "n_leaves": tree.get_n_leaves(),
        "tree_depth": tree.get_depth(),
    }

    report = classification_report(
        y,
        y_pred,
        output_dict=True,
        zero_division=0
    )

    rules_text = export_text(
        tree,
        feature_names=list(X.columns)
    )

    return tree, metrics, report, rules_text


def plot_decision_tree_cluster_explainer(
    tree,
    features_df,
    title="Decision tree explanation of clusters"
):
    plt.figure(figsize=(24, 12))

    plot_tree(
        tree,
        feature_names=list(features_df.columns),
        class_names=[str(c) for c in tree.classes_],
        filled=True,
        rounded=True,
        fontsize=8
    )

    plt.title(title)
    plt.tight_layout()
    plt.show()


def get_decision_tree_feature_importance(tree, features_df):
    importance = pd.Series(
        tree.feature_importances_,
        index=features_df.columns
    ).sort_values(ascending=False)

    return importance[importance > 0]


def plot_tree_feature_importance(tree_feature_importance, top_n=20):
    top_imp = tree_feature_importance.head(top_n).sort_values()

    plt.figure(figsize=(10, max(4, 0.35 * len(top_imp))))
    plt.barh(top_imp.index, top_imp.values)
    plt.xlabel("Decision tree feature importance")
    plt.ylabel("Feature")
    plt.title("Most important features for explaining clusters")
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CELL 10 - Compare clusters with gender-caste membership
# ============================================================

def compare_clusters_with_circles(circle_membership_matrix, labels):
    """
    Compare cluster assignments with an exclusive gender-caste
    interpretation membership matrix.

    Every row must contain exactly one 1, so every node belongs
    to one and only one gender-caste group.
    """

    if circle_membership_matrix.shape[0] != len(labels):
        raise ValueError(
            "Gender-caste membership rows and cluster labels differ."
        )

    row_sums = circle_membership_matrix.sum(axis=1)

    if not (row_sums == 1).all():
        bad_nodes = row_sums[row_sums != 1].index.tolist()[:10]

        raise ValueError(
            "Gender-caste interpretation membership must be exclusive: "
            "every node must belong to exactly one group. "
            f"Example invalid node positions: {bad_nodes}"
        )

    df_cluster_labels = pd.DataFrame(
        {"cluster": labels},
        index=range(1, len(labels) + 1)
    )

    membership_clustered = circle_membership_matrix.join(
        df_cluster_labels,
        how="inner"
    )

    group_counts_per_cluster = membership_clustered.groupby(
        "cluster"
    ).sum()

    group_props_per_cluster = group_counts_per_cluster.div(
        group_counts_per_cluster.sum(axis=1),
        axis=0
    )

    return group_counts_per_cluster, group_props_per_cluster


def plot_circle_composition(
    circle_props_per_cluster,
    top_n_groups=20,
    dataset_name=None
):
    n_groups_available = circle_props_per_cluster.shape[1]
    n_groups_to_plot = min(top_n_groups, n_groups_available)

    selected_groups = (
        circle_props_per_cluster
        .mean(axis=0)
        .sort_values(ascending=False)
        .head(n_groups_to_plot)
        .index
    )

    plot_df = circle_props_per_cluster[selected_groups]

    plot_df.plot(
        kind="bar",
        stacked=True,
        figsize=(14, 7),
        colormap="tab20"
    )

    def get_method_label(dataset_name):
        if "original" in dataset_name:
            return "Original graph"
        elif "method1" in dataset_name:
            return "Multiple encoder"
        elif "method2" in dataset_name:
            return "One encoder"
        elif "method3" in dataset_name:
            return "Batch"
        else:
            return dataset_name


    title = f"{get_method_label(dataset_name)} - Gender-Caste Composition by Cluster"
    plt.title(title)
    plt.xlabel("Cluster")
    plt.ylabel("Proportion of Nodes")
    plt.legend(
        title="Gender-Caste Group",
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CELL 11 - Main clustering analysis function
# ============================================================

def run_clustering_analysis(
    embeddings,
    interpretation_df,
    circle_membership_matrix,
    n_clusters,
    output_folder,
    dataset_name,
    final_method="kmeans",
    random_state=123
):

    print("\n" + "=" * 80)
    print(f"FULL ANALYSIS FOR: {dataset_name}")
    print("=" * 80)

    results, metrics_df = compare_clustering_methods(
        embeddings=embeddings,
        n_clusters=n_clusters,
        random_state=random_state
    )

    print("\n=== Clustering metrics ===")
    print(metrics_df)

    metrics_df.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_metrics_comparison_k{n_clusters}.csv"
        )
    )

    final_result = results[final_method]

    if final_result["labels"] is None:
        raise RuntimeError(
            f"{dataset_name}: {final_method} could not fit the embeddings. "
            f"Status: {final_result['metrics']['fit_status']}"
        )

    final_labels = final_result["labels"]

    effective_clusters = len(np.unique(final_labels))

    labels_df = save_cluster_labels(
        final_labels,
        os.path.join(
            output_folder,
            f"{dataset_name}_{final_method}_labels_k{n_clusters}.csv"
        )
    )

    cluster_counts = plot_cluster_sizes(final_labels)

    cluster_counts.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_cluster_sizes_k{n_clusters}.csv"
        )
    )

    status_df = pd.DataFrame([{
        "dataset": dataset_name,
        "requested_clusters": n_clusters,
        "effective_clusters": effective_clusters,
        "final_method": final_method,
        "analysis_status": (
            "valid"
            if effective_clusters >= 2
            else "collapsed_single_cluster"
        )
    }])

    status_df.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_analysis_status_k{n_clusters}.csv"
        ),
        index=False
    )

    # --------------------------------------------------------
    # Do not force interpretation when all nodes ended up
    # in one effective cluster.
    # --------------------------------------------------------

    if effective_clusters < 2:
        print("\n" + "!" * 80)
        print("CLUSTERING COLLAPSED TO ONE EFFECTIVE CLUSTER")
        print(
            "The embeddings do not support a meaningful multi-cluster "
            "interpretation for this method and feature set."
        )
        print(
            "The notebook saved the labels, cluster size and diagnostics, "
            "but skips feature profiles, mutual information, CLAMP and "
            "decision-tree interpretation."
        )
        print("!" * 80)

        return {
            "status": "collapsed_single_cluster",
            "results": results,
            "metrics_df": metrics_df,
            "labels": final_labels,
            "labels_df": labels_df,
            "cluster_counts": cluster_counts,
            "effective_clusters": effective_clusters,
        }

    # --------------------------------------------------------
    # Valid multi-cluster result
    # --------------------------------------------------------

    if final_method == "kmeans":
        centroides = results["kmeans"]["centroides"]

        centroides_df = pd.DataFrame(
            centroides,
            columns=[
                f"embedding_dim_{i}"
                for i in range(centroides.shape[1])
            ]
        )

        centroides_df.index.name = "cluster"

        print("\n=== KMeans centroids ===")
        print(centroides_df)

        centroides_df.to_csv(
            os.path.join(
                output_folder,
                f"{dataset_name}_kmeans_centroids_k{n_clusters}.csv"
            )
        )

    emb_2d = compute_tsne(
        embeddings,
        random_state=random_state
    )

    plot_tsne_clusters(
        emb_2d,
        final_labels,
        title=f"{dataset_name} - {final_method} - k={n_clusters}"
    )

    plot_feature_sparsity(interpretation_df)

    cluster_feature_profiles = compute_cluster_feature_profiles(
        interpretation_df,
        final_labels
    )

    print("\n=== Cluster feature profiles ===")
    print(cluster_feature_profiles)

    cluster_feature_profiles.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_cluster_feature_profiles_k{n_clusters}.csv"
        )
    )

    plot_feature_profile_heatmap(cluster_feature_profiles)

    sorted_features = print_features_above_threshold(
        cluster_feature_profiles,
        threshold=0.3
    )

    sorted_features.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_features_above_threshold_k{n_clusters}.csv"
        )
    )

    mi_scores = compute_mutual_information_features(
        interpretation_df,
        final_labels
    )

    print("\n=== Top mutual information features ===")
    print(mi_scores.head(20))

    mi_scores.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_mutual_information_scores_k{n_clusters}.csv"
        )
    )

    plot_top_features_per_cluster(
        cluster_feature_profiles,
        threshold=0.4
    )

    clamp_prototypes = compute_clamp_feature_prototypes(
        interpretation_df,
        final_labels,
        lower_q=0.10,
        upper_q=0.90,
        top_n=15
    )

    print("\n=== CLAMP prototypes ===")
    print(clamp_prototypes.head(30))

    clamp_prototypes.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_clamp_prototypes_k{n_clusters}.csv"
        ),
        index=False
    )

    plot_clamp_feature_prototypes(
        clamp_prototypes,
        top_n=10,
        value_col="cluster_prevalence"
    )

    clamp_heatmap = plot_clamp_heatmap(
        clamp_prototypes,
        top_n_features=30
    )

    clamp_heatmap.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_clamp_heatmap_k{n_clusters}.csv"
        )
    )

    tree, tree_metrics, tree_report, tree_rules = (
        train_decision_tree_cluster_explainer(
            interpretation_df,
            final_labels,
            max_depth=4,
            min_samples_leaf=10,
            random_state=random_state
        )
    )

    print("\n=== Decision tree fidelity metrics ===")
    print(pd.DataFrame([tree_metrics]))

    print("\n=== Decision tree rules ===")
    print(tree_rules)

    plot_decision_tree_cluster_explainer(
        tree,
        interpretation_df,
        title=f"{dataset_name} - Decision Tree"
    )

    tree_feature_importance = get_decision_tree_feature_importance(
        tree,
        interpretation_df
    )

    print("\n=== Decision tree feature importance ===")
    print(tree_feature_importance.head(20))

    tree_feature_importance.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_tree_feature_importance_k{n_clusters}.csv"
        )
    )

    plot_tree_feature_importance(
        tree_feature_importance,
        top_n=20
    )

    circle_counts, circle_props = compare_clusters_with_circles(
        circle_membership_matrix,
        final_labels
    )

    print("\n=== Circle proportions by cluster ===")
    print(circle_props)

    circle_counts.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_circle_counts_k{n_clusters}.csv"
        )
    )

    circle_props.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_circle_props_k{n_clusters}.csv"
        )
    )

    plot_circle_composition(
    circle_props,
    top_n_groups=20,
    dataset_name=dataset_name
)

    return {
        "status": "valid",
        "results": results,
        "metrics_df": metrics_df,
        "labels": final_labels,
        "labels_df": labels_df,
        "cluster_counts": cluster_counts,
        "effective_clusters": effective_clusters,
        "cluster_feature_profiles": cluster_feature_profiles,
        "sorted_features": sorted_features,
        "mi_scores": mi_scores,
        "clamp_prototypes": clamp_prototypes,
        "clamp_heatmap": clamp_heatmap,
        "tree": tree,
        "tree_metrics": tree_metrics,
        "tree_report": tree_report,
        "tree_rules": tree_rules,
        "tree_feature_importance": tree_feature_importance,
        "circle_counts": circle_counts,
        "circle_props": circle_props,
    }

In [ ]:
# ============================================================
# CELL 12 - Paths and global parameters
# ============================================================

dgi_folder = "village51_dgi"

# Interpretation matrix for all-features embeddings
features_path = "features_interpretation_village51.csv"

# Interpretation matrix for ARD circles-only embeddings
ard_path = "ard_matrix_village51.csv"

# Original operational membership:
# caste and gender remain separate for ARD and NSUM.
membership_original_path = "membership_matrix_village51.csv"

# New output folder: keeps previous clustering outputs intact.
output_folder = (
    "village51_clustering_training_embeddings_comparison"
)

os.makedirs(output_folder, exist_ok=True)

In [ ]:
# ============================================================
# CELL 13 - Load interpretation data and build gender-caste matrix
# ============================================================

# ------------------------------------------------------------
# ARD defines the canonical original-node order.
# ------------------------------------------------------------

ard_df, original_node_ids = load_matrix_for_synthetic_order(
    path=ard_path,
    reference_node_ids=None,
    matrix_name="ARD"
)

# ------------------------------------------------------------
# All-features interpretation matrix aligned to ARD order.
# ------------------------------------------------------------

features_interpretation_df, features_node_ids = (
    load_matrix_for_synthetic_order(
        path=features_path,
        reference_node_ids=original_node_ids,
        matrix_name="features_interpretation"
    )
)

# ------------------------------------------------------------
# Original membership aligned to ARD order.
# This matrix is not used as DGI input anymore.
# It is only used here to construct gender × caste groups.
# ------------------------------------------------------------

membership_original_df, membership_node_ids = (
    load_matrix_for_synthetic_order(
        path=membership_original_path,
        reference_node_ids=original_node_ids,
        matrix_name="original_membership"
    )
)

assert features_node_ids == original_node_ids, (
    "Features interpretation matrix is not aligned with ARD."
)

assert membership_node_ids == original_node_ids, (
    "Original membership matrix is not aligned with ARD."
)


def sanitize_caste_label(caste_column):
    """
    Converts source membership columns into readable English labels.

    Example:
    caste_SCHEDULED CASTE -> caste_scheduled_caste
    """

    label = caste_column.replace("caste_", "", 1).strip().lower()

    label = (
        label
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
    )

    while "__" in label:
        label = label.replace("__", "_")

    return f"caste_{label}"


def build_gender_caste_interpretation_membership(membership_df):
    """
    Builds an exclusive gender-caste membership matrix.

    Each node belongs to exactly one category:
    male/female × caste category.

    This matrix is only for post-hoc cluster interpretation.
    It does not replace the operational membership matrix used
    with ARD in Steps 1 and 2.
    """

    # IMPORTANT:
    # Confirm this coding against the codebook.
    gender_column_to_label = {
        "resp_gend_1.0": "male",
        "resp_gend_2.0": "female",
    }

    expected_gender_columns = list(gender_column_to_label.keys())

    missing_gender_columns = [
        col for col in expected_gender_columns
        if col not in membership_df.columns
    ]

    if missing_gender_columns:
        raise ValueError(
            "Expected gender membership columns not found: "
            f"{missing_gender_columns}"
        )

    caste_columns = [
        col for col in membership_df.columns
        if col.startswith("caste_")
    ]

    if len(caste_columns) != 4:
        raise ValueError(
            "Expected exactly four caste membership columns, but found: "
            f"{caste_columns}"
        )

    preferred_caste_order = [
        "caste_GENERAL",
        "caste_OBC",
        "caste_SCHEDULED CASTE",
        "caste_SCHEDULED TRIBE",
    ]

    if set(caste_columns) == set(preferred_caste_order):
        caste_columns = preferred_caste_order
    else:
        caste_columns = sorted(caste_columns)

    caste_membership = membership_df[caste_columns]
    gender_membership = membership_df[expected_gender_columns]

    caste_row_sums = caste_membership.sum(axis=1)
    gender_row_sums = gender_membership.sum(axis=1)

    if not (caste_row_sums == 1).all():
        bad_nodes = caste_row_sums[
            caste_row_sums != 1
        ].index.tolist()[:10]

        raise ValueError(
            "Each node must belong to exactly one caste category. "
            f"Invalid node positions: {bad_nodes}"
        )

    if not (gender_row_sums == 1).all():
        bad_nodes = gender_row_sums[
            gender_row_sums != 1
        ].index.tolist()[:10]

        raise ValueError(
            "Each node must belong to exactly one gender category. "
            f"Invalid node positions: {bad_nodes}"
        )

    gender_source = gender_membership.idxmax(axis=1)
    caste_source = caste_membership.idxmax(axis=1)

    gender_labels = gender_source.map(gender_column_to_label)

    caste_column_to_label = {
        caste_column: sanitize_caste_label(caste_column)
        for caste_column in caste_columns
    }

    caste_labels = caste_source.map(caste_column_to_label)

    combined_group = gender_labels + "_" + caste_labels

    desired_order = []

    for gender in ["male", "female"]:
        for caste_column in caste_columns:
            desired_order.append(
                f"{gender}_{caste_column_to_label[caste_column]}"
            )

    gender_caste_df = pd.get_dummies(
        combined_group,
        dtype=int
    )

    gender_caste_df = gender_caste_df.reindex(
        columns=desired_order,
        fill_value=0
    )

    gender_caste_df.index = membership_df.index
    gender_caste_df.index.name = "node_position"

    assert (gender_caste_df.sum(axis=1) == 1).all(), (
        "Every node must belong to exactly one gender-caste group."
    )

    return gender_caste_df


gender_caste_membership_df = (
    build_gender_caste_interpretation_membership(
        membership_original_df
    )
)

# Save with original village node IDs for traceability.
gender_caste_membership_original_ids = (
    gender_caste_membership_df.copy()
)

gender_caste_membership_original_ids.index = original_node_ids
gender_caste_membership_original_ids.index.name = "node_id"

gender_caste_membership_original_ids.to_csv(
    os.path.join(
        output_folder,
        "membership_matrix_gender_caste_interpretation_village51.csv"
    )
)

gender_caste_summary_df = pd.DataFrame({
    "num_nodes": gender_caste_membership_df.sum(axis=0)
})

gender_caste_summary_df["percent_nodes"] = (
    gender_caste_summary_df["num_nodes"]
    / len(gender_caste_membership_df)
    * 100
).round(2)

gender_caste_summary_df.to_csv(
    os.path.join(
        output_folder,
        "gender_caste_interpretation_summary_village51.csv"
    )
)

print("Features interpretation shape:", features_interpretation_df.shape)
print("ARD interpretation shape:", ard_df.shape)
print("Original membership shape:", membership_original_df.shape)
print(
    "Gender-caste interpretation membership shape:",
    gender_caste_membership_df.shape
)

print("\nFeatures preview:")
display(features_interpretation_df.head())

print("\nARD preview:")
display(ard_df.head())

print("\nGender-caste membership preview:")
display(gender_caste_membership_df.head())

print("\nGender-caste membership summary:")
display(gender_caste_summary_df)

print("\nGender-caste columns:")
print(list(gender_caste_membership_df.columns))

In [ ]:
# ============================================================
# CELL 14 - Dataset registry
# ============================================================

dataset_registry = {
    # --------------------------------------------------------
    # Original graph
    # --------------------------------------------------------
    "original_all_features": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_original_all_features.npy"
        ),
        "interpretation_df": features_interpretation_df,
        "interpretation_note": (
            "DGI input: individual demographic features plus graph "
            "structure."
        ),
    },
    "original_ard_circles_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_original_ard_circles_only.npy"
        ),
        "interpretation_df": ard_df,
        "interpretation_note": (
            "DGI input: ARD counts for caste and gender groups plus "
            "graph structure."
        ),
    },
    "original_graph_structure_identity_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_original_graph_structure_identity_only.npy"
        ),
        "interpretation_df": features_interpretation_df,
        "interpretation_note": (
            "DGI input: identity matrix and graph structure only. "
            "Demographic features are used exclusively for post-hoc "
            "interpretation."
        ),
    },

    # --------------------------------------------------------
    # Method 1 - Independent DGI ensemble
    # --------------------------------------------------------
    "method1_independent_mean_all_features": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method1_independent_mean_all_features.npy"
        ),
        "interpretation_df": features_interpretation_df,
        "interpretation_note": (
            "DGI input: individual demographic features plus graph "
            "structure."
        ),
    },
    "method1_independent_mean_ard_circles_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method1_independent_mean_ard_circles_only.npy"
        ),
        "interpretation_df": ard_df,
        "interpretation_note": (
            "DGI input: ARD counts for caste and gender groups plus "
            "graph structure."
        ),
    },
    "method1_independent_mean_graph_structure_identity_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method1_independent_mean_"
            "graph_structure_identity_only.npy"
        ),
        "interpretation_df": features_interpretation_df,
        "interpretation_note": (
            "DGI input: identity matrix and graph structure only. "
            "Demographic features are used exclusively for post-hoc "
            "interpretation."
        ),
    },

    # --------------------------------------------------------
    # Method 2 - Shared sequential DGI
    # --------------------------------------------------------
    "method2_shared_sequential_mean_all_features": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method2_shared_sequential_mean_all_features.npy"
        ),
        "interpretation_df": features_interpretation_df,
        "interpretation_note": (
            "DGI input: individual demographic features plus graph "
            "structure."
        ),
    },
    "method2_shared_sequential_mean_ard_circles_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method2_shared_sequential_mean_"
            "ard_circles_only.npy"
        ),
        "interpretation_df": ard_df,
        "interpretation_note": (
            "DGI input: ARD counts for caste and gender groups plus "
            "graph structure."
        ),
    },
    "method2_shared_sequential_mean_graph_structure_identity_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method2_shared_sequential_mean_"
            "graph_structure_identity_only.npy"
        ),
        "interpretation_df": features_interpretation_df,
        "interpretation_note": (
            "DGI input: identity matrix and graph structure only. "
            "Demographic features are used exclusively for post-hoc "
            "interpretation."
        ),
    },

    # --------------------------------------------------------
    # Method 3 - Batch DGI
    # --------------------------------------------------------
    "method3_batch_mean_all_features": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method3_batch_mean_all_features.npy"
        ),
        "interpretation_df": features_interpretation_df,
        "interpretation_note": (
            "DGI input: individual demographic features plus graph "
            "structure."
        ),
    },
    "method3_batch_mean_ard_circles_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method3_batch_mean_ard_circles_only.npy"
        ),
        "interpretation_df": ard_df,
        "interpretation_note": (
            "DGI input: ARD counts for caste and gender groups plus "
            "graph structure."
        ),
    },
    "method3_batch_mean_graph_structure_identity_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method3_batch_mean_"
            "graph_structure_identity_only.npy"
        ),
        "interpretation_df": features_interpretation_df,
        "interpretation_note": (
            "DGI input: identity matrix and graph structure only. "
            "Demographic features are used exclusively for post-hoc "
            "interpretation."
        ),
    },
}

# ------------------------------------------------------------
# Additional training-time embedding variants
# ------------------------------------------------------------

dataset_registry.update({
    # --------------------------------------------------------
    # Original graph - training-time embeddings
    # --------------------------------------------------------
    "original_TRAINING_all_features": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_original_TRAINING_all_features.npy"
        ),
        "interpretation_df": features_interpretation_df,
        "interpretation_note": (
            "Original-graph embeddings taken directly during training; "
            "DGI input: individual demographic features plus graph structure."
        ),
    },

    "original_TRAINING_ard_circles_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_original_TRAINING_ard_circles_only.npy"
        ),
        "interpretation_df": ard_df,
        "interpretation_note": (
            "Original-graph embeddings taken directly during training; "
            "DGI input: ARD circles plus graph structure."
        ),
    },

    "original_TRAINING_graph_structure_identity_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_original_TRAINING_"
            "graph_structure_identity_only.npy"
        ),
        "interpretation_df": features_interpretation_df,
        "interpretation_note": (
            "Original-graph embeddings taken directly during training; "
            "DGI input: identity matrix and graph structure only."
        ),
    },

    "method2_shared_sequential_TRAINING_MEAN_all_features": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method2_shared_sequential_TRAINING_MEAN_"
            "all_features.npy"
        ),
        "interpretation_df": features_interpretation_df,
        "interpretation_note": (
            "One Encoder embeddings taken directly during training; "
            "DGI input: individual demographic features plus graph structure."
        ),
    },

    "method2_shared_sequential_TRAINING_MEAN_ard_circles_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method2_shared_sequential_TRAINING_MEAN_"
            "ard_circles_only.npy"
        ),
        "interpretation_df": ard_df,
        "interpretation_note": (
            "One Encoder embeddings taken directly during training; "
            "DGI input: ARD circles plus graph structure."
        ),
    },

    "method2_shared_sequential_TRAINING_MEAN_graph_structure_identity_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method2_shared_sequential_TRAINING_MEAN_"
            "graph_structure_identity_only.npy"
        ),
        "interpretation_df": features_interpretation_df,
        "interpretation_note": (
            "One Encoder embeddings taken directly during training; "
            "DGI input: identity matrix and graph structure only."
        ),
    },

    "method3_batch_TRAINING_MEAN_all_features": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method3_batch_TRAINING_MEAN_all_features.npy"
        ),
        "interpretation_df": features_interpretation_df,
        "interpretation_note": (
            "Batch embeddings taken directly during training; "
            "DGI input: individual demographic features plus graph structure."
        ),
    },

    "method3_batch_TRAINING_MEAN_ard_circles_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method3_batch_TRAINING_MEAN_ard_circles_only.npy"
        ),
        "interpretation_df": ard_df,
        "interpretation_note": (
            "Batch embeddings taken directly during training; "
            "DGI input: ARD circles plus graph structure."
        ),
    },

    "method3_batch_TRAINING_MEAN_graph_structure_identity_only": {
        "embedding_path": os.path.join(
            dgi_folder,
            "embeddings_method3_batch_TRAINING_MEAN_"
            "graph_structure_identity_only.npy"
        ),
        "interpretation_df": features_interpretation_df,
        "interpretation_note": (
            "Batch embeddings taken directly during training; "
            "DGI input: identity matrix and graph structure only."
        ),
    },
})


In [ ]:
# ============================================================
# CELL 15 - Helper to load and check one dataset
# ============================================================

def load_dataset_from_registry(dataset_name):
    dataset_info = dataset_registry[dataset_name]

    embedding_path = dataset_info["embedding_path"]
    interpretation_df = dataset_info["interpretation_df"]

    interpretation_note = dataset_info.get(
        "interpretation_note",
        "No interpretation note available."
    )

    if not os.path.exists(embedding_path):
        raise FileNotFoundError(
            f"Missing embedding file: {embedding_path}"
        )

    embeddings = load_embeddings(embedding_path)

    assert embeddings.shape[0] == interpretation_df.shape[0], (
        f"{dataset_name}: embeddings/interpretation mismatch"
    )

    assert embeddings.shape[0] == gender_caste_membership_df.shape[0], (
        f"{dataset_name}: embeddings/gender-caste membership mismatch"
    )

    print(f"Dataset loaded: {dataset_name}")
    print("Embeddings shape:", embeddings.shape)
    print("Interpretation shape:", interpretation_df.shape)
    print(
        "Gender-caste membership shape:",
        gender_caste_membership_df.shape
    )
    print("\nInterpretation note:")
    print(interpretation_note)

    return embeddings, interpretation_df

In [ ]:
# ============================================================
# CELL 16 - Helper to run diagnostics for one dataset
# ============================================================

def run_diagnostics_for_dataset(dataset_name, embeddings):
    print(f"\nRunning diagnostics for: {dataset_name}")

    diagnostics_df = compute_clustering_diagnostics(
        embeddings,
        k_min=2,
        k_max=8,
        random_state=123
    )

    diagnostics_df["dataset"] = dataset_name

    diagnostics_df.to_csv(
        os.path.join(
            output_folder,
            f"{dataset_name}_all_diagnostics_k2_k8.csv"
        ),
        index=False
    )

    split_diagnostics_by_method(
        diagnostics_df=diagnostics_df,
        output_folder=output_folder,
        dataset_name=dataset_name
    )

    return diagnostics_df

In [ ]:
# ============================================================
# CELL 17 - Helper to run final analysis for one dataset
# ============================================================

def run_final_dataset_analysis(
    dataset_name,
    n_clusters=2,
    final_method="kmeans",
    random_state=123
):
    # Ensure the output folder exists before diagnostics or
    # clustering files are written.
    os.makedirs(output_folder, exist_ok=True)

    embeddings, interpretation_df = load_dataset_from_registry(
        dataset_name
    )

    diagnostics = run_diagnostics_for_dataset(
        dataset_name,
        embeddings
    )

    result = run_clustering_analysis(
        embeddings=embeddings,
        interpretation_df=interpretation_df,
        circle_membership_matrix=gender_caste_membership_df,
        n_clusters=n_clusters,
        output_folder=output_folder,
        dataset_name=dataset_name,
        final_method=final_method,
        random_state=random_state
    )

    return diagnostics, result

In [ ]:
# ============================================================
# CELL 18 - MODEL 0A - Original graph with all features
# ============================================================

diagnostics_original_all, result_original_all = run_final_dataset_analysis(
    dataset_name="original_all_features",
    n_clusters=8,
    final_method="kmeans",
    random_state=123
)

In [ ]:
# ============================================================
# CELL 19 - MODEL 0B - Original graph with ARD circles-only
# ============================================================

diagnostics_original_ard, result_original_ard = run_final_dataset_analysis(
    dataset_name="original_ard_circles_only",
    n_clusters=8,
    final_method="kmeans",
    random_state=123
)

In [ ]:
# ============================================================
# CELL 20 - MODEL 0C - Original graph with structure only
# ============================================================

diagnostics_original_structure, result_original_structure = (
    run_final_dataset_analysis(
        dataset_name="original_graph_structure_identity_only",
        n_clusters=8,
        final_method="kmeans",
        random_state=123
    )
)

In [ ]:
# ============================================================
# TRAINING-TIME MODEL 0A - Original graph with all features
# ============================================================

diagnostics_original_training_all, result_original_training_all = (
    run_final_dataset_analysis(
        dataset_name="original_TRAINING_all_features",
        n_clusters=8,
        final_method="kmeans",
        random_state=123
    )
)


In [ ]:
# ============================================================
# TRAINING-TIME MODEL 0B - Original graph with ARD circles-only
# ============================================================

diagnostics_original_training_ard, result_original_training_ard = (
    run_final_dataset_analysis(
        dataset_name="original_TRAINING_ard_circles_only",
        n_clusters=8,
        final_method="kmeans",
        random_state=123
    )
)


In [ ]:
# ============================================================
# TRAINING-TIME MODEL 0C - Original graph with structure only
# ============================================================

diagnostics_original_training_structure, result_original_training_structure = (
    run_final_dataset_analysis(
        dataset_name="original_TRAINING_graph_structure_identity_only",
        n_clusters=8,
        final_method="kmeans",
        random_state=123
    )
)


In [ ]:
# ============================================================
# CELL 21 - MODEL 1A - Independent DGI ensemble with all features
# ============================================================

diagnostics_method1_all, result_method1_all = run_final_dataset_analysis(
    dataset_name="method1_independent_mean_all_features",
    n_clusters=8,
    final_method="kmeans",
    random_state=123
)

In [ ]:
# ============================================================
# CELL 22 - MODEL 1B - Independent DGI ensemble with ARD circles-only
# ============================================================

diagnostics_method1_ard, result_method1_ard = run_final_dataset_analysis(
    dataset_name="method1_independent_mean_ard_circles_only",
    n_clusters=8,
    final_method="kmeans",
    random_state=123
)

In [ ]:
# ============================================================
# CELL 23 - MODEL 1C - Independent DGI ensemble with structure only
# ============================================================

diagnostics_method1_structure, result_method1_structure = (
    run_final_dataset_analysis(
        dataset_name=(
            "method1_independent_mean_"
            "graph_structure_identity_only"
        ),
        n_clusters=8,
        final_method="kmeans",
        random_state=123
    )
)

In [ ]:
# ============================================================
# CELL 24 - MODEL 2A - Shared sequential DGI with all features
# ============================================================

diagnostics_method2_all, result_method2_all = run_final_dataset_analysis(
    dataset_name="method2_shared_sequential_mean_all_features",
    n_clusters=8,
    final_method="kmeans",
    random_state=123
)

In [ ]:
# ============================================================
# CELL 25 - MODEL 2B - Shared sequential DGI with ARD circles-only
# ============================================================

diagnostics_method2_ard, result_method2_ard = run_final_dataset_analysis(
    dataset_name="method2_shared_sequential_mean_ard_circles_only",
    n_clusters=8,
    final_method="kmeans",
    random_state=123
)

In [ ]:
# ============================================================
# DISABLE SAVING - Run analysis without creating CSV files
# ============================================================

SAVE_OUTPUTS = False

if not SAVE_OUTPUTS:

    def skip_dataframe_to_csv(self, *args, **kwargs):
        return None

    def skip_series_to_csv(self, *args, **kwargs):
        return None

    pd.DataFrame.to_csv = skip_dataframe_to_csv
    pd.Series.to_csv = skip_series_to_csv

    print("Saving CSV files is disabled for this notebook session.")

In [ ]:
# ============================================================
# CELL 26 - MODEL 2C - Shared sequential DGI with structure only
# ============================================================

diagnostics_method2_structure, result_method2_structure = (
    run_final_dataset_analysis(
        dataset_name=(
            "method2_shared_sequential_mean_"
            "graph_structure_identity_only"
        ),
        n_clusters=8,
        final_method="kmeans",
        random_state=123
    )
)

In [ ]:
# ============================================================
# CELL 27 - MODEL 3A - Batch DGI with all features
# ============================================================

diagnostics_method3_all, result_method3_all = run_final_dataset_analysis(
    dataset_name="method3_batch_mean_all_features",
    n_clusters=8,
    final_method="kmeans",
    random_state=123
)

In [ ]:
# ============================================================
# CELL 28 - MODEL 3B - Batch DGI with ARD circles-only
# ============================================================

diagnostics_method3_ard, result_method3_ard = run_final_dataset_analysis(
    dataset_name="method3_batch_mean_ard_circles_only",
    n_clusters=8,
    final_method="kmeans",
    random_state=123
)

In [ ]:
# ============================================================
# CELL 29 - MODEL 3C - Batch DGI with structure only
# ============================================================

diagnostics_method3_structure, result_method3_structure = (
    run_final_dataset_analysis(
        dataset_name=(
            "method3_batch_mean_"
            "graph_structure_identity_only"
        ),
        n_clusters=8,
        final_method="kmeans",
        random_state=123
    )
)

In [ ]:
# ============================================================
# TRAINING-TIME MODEL 2A - One Encoder with all features
# ============================================================

diagnostics_method2_training_all, result_method2_training_all = (
    run_final_dataset_analysis(
        dataset_name=(
            "method2_shared_sequential_TRAINING_MEAN_all_features"
        ),
        n_clusters=8,
        final_method="kmeans",
        random_state=123
    )
)


In [ ]:
# ============================================================
# TRAINING-TIME MODEL 2B - One Encoder with ARD circles-only
# ============================================================

diagnostics_method2_training_ard, result_method2_training_ard = (
    run_final_dataset_analysis(
        dataset_name=(
            "method2_shared_sequential_TRAINING_MEAN_ard_circles_only"
        ),
        n_clusters=8,
        final_method="kmeans",
        random_state=123
    )
)


In [ ]:
# ============================================================
# TRAINING-TIME MODEL 2C - One Encoder with structure only
# ============================================================

diagnostics_method2_training_structure, result_method2_training_structure = (
    run_final_dataset_analysis(
        dataset_name=(
            "method2_shared_sequential_TRAINING_MEAN_"
            "graph_structure_identity_only"
        ),
        n_clusters=8,
        final_method="kmeans",
        random_state=123
    )
)


In [ ]:
# ============================================================
# TRAINING-TIME MODEL 3A - Batch with all features
# ============================================================

diagnostics_method3_training_all, result_method3_training_all = (
    run_final_dataset_analysis(
        dataset_name="method3_batch_TRAINING_MEAN_all_features",
        n_clusters=8,
        final_method="kmeans",
        random_state=123
    )
)


In [ ]:
# ============================================================
# TRAINING-TIME MODEL 3B - Batch with ARD circles-only
# ============================================================

diagnostics_method3_training_ard, result_method3_training_ard = (
    run_final_dataset_analysis(
        dataset_name="method3_batch_TRAINING_MEAN_ard_circles_only",
        n_clusters=8,
        final_method="kmeans",
        random_state=123
    )
)


In [ ]:
# ============================================================
# TRAINING-TIME MODEL 3C - Batch with structure only
# ============================================================

diagnostics_method3_training_structure, result_method3_training_structure = (
    run_final_dataset_analysis(
        dataset_name=(
            "method3_batch_TRAINING_MEAN_"
            "graph_structure_identity_only"
        ),
        n_clusters=8,
        final_method="kmeans",
        random_state=123
    )
)


In [ ]:
# ============================================================
# FINAL COMPARISON
# Post-training vs training-time embeddings
# ============================================================

comparison_pairs = [
    (
        "Original Graph",
        "all_features",
        result_original_all,
        result_original_training_all
    ),
    (
        "Original Graph",
        "ard_circles_only",
        result_original_ard,
        result_original_training_ard
    ),
    (
        "Original Graph",
        "graph_structure_identity_only",
        result_original_structure,
        result_original_training_structure
    ),
    (
        "One Encoder",
        "all_features",
        result_method2_all,
        result_method2_training_all
    ),
    (
        "One Encoder",
        "ard_circles_only",
        result_method2_ard,
        result_method2_training_ard
    ),
    (
        "One Encoder",
        "graph_structure_identity_only",
        result_method2_structure,
        result_method2_training_structure
    ),
    (
        "Batch",
        "all_features",
        result_method3_all,
        result_method3_training_all
    ),
    (
        "Batch",
        "ard_circles_only",
        result_method3_ard,
        result_method3_training_ard
    ),
    (
        "Batch",
        "graph_structure_identity_only",
        result_method3_structure,
        result_method3_training_structure
    ),
]

comparison_rows = []

for (
    model_name,
    feature_set,
    post_result,
    training_result
) in comparison_pairs:

    post_metrics = (
        post_result["results"]["kmeans"]["metrics"]
    )

    training_metrics = (
        training_result["results"]["kmeans"]["metrics"]
    )

    comparison_rows.append({
        "model": model_name,
        "feature_set": feature_set,
        "embedding_source": "post_training_reencoded",
        "silhouette": post_metrics["silhouette"],
        "calinski_harabasz": post_metrics["calinski_harabasz"],
        "davies_bouldin": post_metrics["davies_bouldin"],
        "n_clusters": post_metrics["n_clusters"],
        "fit_status": post_metrics["fit_status"],
    })

    comparison_rows.append({
        "model": model_name,
        "feature_set": feature_set,
        "embedding_source": "training_time",
        "silhouette": training_metrics["silhouette"],
        "calinski_harabasz": training_metrics["calinski_harabasz"],
        "davies_bouldin": training_metrics["davies_bouldin"],
        "n_clusters": training_metrics["n_clusters"],
        "fit_status": training_metrics["fit_status"],
    })

training_vs_post_comparison = pd.DataFrame(
    comparison_rows
)

print("\n" + "=" * 90)
print(
    "INDIAN: POST-TRAINING VS TRAINING-TIME "
    "EMBEDDING COMPARISON"
)
print("=" * 90)

print(training_vs_post_comparison)

# Respect the original notebook's SAVE_OUTPUTS setting.
if globals().get("SAVE_OUTPUTS", True):
    comparison_path = os.path.join(
        output_folder,
        "training_vs_post_training_clustering_comparison.csv"
    )

    training_vs_post_comparison.to_csv(
        comparison_path,
        index=False
    )

    print(
        "\nSaved comparison table to:",
        comparison_path
    )
else:
    print(
        "\nCSV saving remains disabled, following the original "
        "Step 4 configuration."
    )
